# **NEUROCHIP SMARTSILICON**
### Smart GNN-based framework for VLSI fault detection and localization

---
# Data Extractor Notebook
---

### 1. Load the ESCAD OpenRTLSet VLSI Dataset

Download the dataset and store it as a CSV file

In [1]:
import warnings
warnings.filterwarnings("ignore")

from datasets import load_dataset

# Load the dataset and store it locally as a CSV file
data = load_dataset("ESCAD/OpenRTLSet", split="train")
data.to_csv("escad_openrtlset.csv", index=False)

Creating CSV from Arrow format: 100%|██████████| 132/132 [00:15<00:00,  8.51ba/s]


1007264900

---
### 2. Circuit Type Categorization

Categorize the dataset into 4 types:
- Arithmetic circuits
- Memory circuits
- Control and Logic circuits
- Miscellaneous circuits

Configuration for categorization

In [2]:
CSV_PATH   = "escad_openrtlset.csv"
OUTPUT_DIR = "escad_openrtlset_categorized"

# Keyword List for categorization
CATEGORY_KEYWORDS = {
 
    # ── Arithmetic Modules ────────────────────────────────────────────────────
    # Adders, subtractors, multipliers, dividers, accumulators, comparators
    # Evidence from corpus: "arithmetic operation"(23), "addition"(27),
    # "subtraction"(4), "xor reduction"(4), "overflow"(5), "sign extension"(3)
    "arithmetic": [
        # Component names — strongest signals
        (r"\bfull[\s\-]?adder\b",                 3),
        (r"\bhalf[\s\-]?adder\b",                 3),
        (r"\badder\b",                             3),
        (r"\bsubtractor\b",                        3),
        (r"\bmultiplier\b",                        3),
        (r"\bdivider\b",                           3),
        (r"\baccumulator\b",                       3),
        (r"\bcomparator\b",                        3),
        (r"\balu\b",                               3),
        (r"\barithmetic[\s\-]?logic[\s\-]?unit\b", 3),
        # Operation keywords
        (r"\barithmetic[\s\w]*operation",          2),
        (r"\baddition\b",                          2),
        (r"\bsubtraction\b",                       2),
        (r"\bmultiplication\b",                    2),
        (r"\bdivision\b",                          2),
        (r"\bincrement\b",                         2),
        (r"\bdecrement\b",                         2),
        (r"\boverflow\b",                          2),
        (r"\bcarry[\s\-]?out\b",                   2),
        (r"\bcarry[\s\-]?in\b",                    2),
        (r"\bcarry\b",                             1),
        (r"\bquotient\b",                          2),
        (r"\bremainder\b",                         2),
        (r"\bmodulo\b",                            2),
        (r"\bpopcount\b",                          3),
        (r"\bparity\b",                            2),
        (r"\bxor[\s\-]?reduction\b",               3),
        (r"\bsign[\s\-]?extension\b",              2),
        (r"\bsign[\s\-]?extend\b",                 2),
        (r"\bfloating[\s\-]?point\b",              3),
        (r"\bfixed[\s\-]?point\b",                 2),
        (r"\btwo[\s\-]?stage[\s\-]?pipeline\b",    2),
        (r"\bpipelined.*arithmetic\b",             2),
        (r"\barithmetic.*pipeline\b",              2),
        (r"\bsum\b",                               1),
        (r"\boperand\b",                           2),
        (r"\bsigned.*arithmetic\b",                2),
        (r"\bunsigned.*arithmetic\b",              2),
        (r"\bbit[\s\-]?width\b",                   1),
        (r"\bnumerical\b",                         1),
        (r"\bcompute\b",                           1),
        (r"\bcalculation\b",                       1),
        (r"\bsubtract\b",                          2),
        (r"\bmultiply\b",                          2),
        (r"\bdivide\b",                            2),
        (r"\baccumulate\b",                        2),
    ],
 
    # ── Memory & Storage Modules ──────────────────────────────────────────────
    # FIFO, LIFO, RAM, ROM, shift registers, LFSR, barrel shifters, buffers
    # Evidence: "ram"(218), "rom"(112), "fifo"(13), "shift register"(12),
    # "write enable"(10), "data storage"(8), "memory array"(6)
    "memory_storage": [
        # Named structures — highest confidence
        (r"\bfifo\b",                              3),
        (r"\blifo\b",                              3),
        (r"\bsram\b",                              3),
        (r"\bdram\b",                              3),
        (r"\beeprom\b",                            3),
        (r"\blfsr\b",                              3),
        (r"\blinear[\s\-]?feedback[\s\-]?shift[\s\-]?register\b", 3),
        (r"\bbarrel[\s\-]?shift",                  3),
        (r"\bregister[\s\-]?file\b",               3),
        (r"\bregfile\b",                           3),
        (r"\bdual[\s\-]?port\b",                   3),
        # RAM / ROM (very common — use context to avoid false positives)
        (r"\bram\b",                               2),
        (r"\brom\b",                               2),
        (r"\bmemory[\s\w]*array\b",                3),
        (r"\bmemory[\s\w]*block\b",                3),
        (r"\bmemory[\s\w]*module\b",               2),
        (r"\bmemory\b",                            1),
        (r"\bshift[\s\-]?register\b",              3),
        (r"\bdata[\s\-]?storage\b",                2),
        (r"\btemporary[\s\w]*storage\b",           2),
        (r"\blong[\s\-]?term[\s\w]*storage\b",     2),
        (r"\bwrite[\s\-]?enable\b",                2),
        (r"\bread[\s\-]?enable\b",                 2),
        (r"\bread[\s\-]?write\b",                  2),
        (r"\baddress[\s\w]*width\b",               2),
        (r"\baddress[\s\w]*space\b",               2),
        (r"\bfirst[\s\-]?in.*first[\s\-]?out\b",   3),
        (r"\blast[\s\-]?in.*first[\s\-]?out\b",    3),
        (r"\bfull[\s\-]?flag\b",                   2),
        (r"\bempty[\s\-]?flag\b",                  2),
        (r"\bread[\s\-]?pointer\b",                2),
        (r"\bwrite[\s\-]?pointer\b",               2),
        (r"\bsynchronous.*memory\b",               2),
        (r"\basynchronous.*memory\b",              2),
        (r"\bserial[\s\-]?to[\s\-]?parallel\b",    2),
        (r"\bparallel[\s\-]?to[\s\-]?serial\b",    2),
        (r"\bserial.*parallel\b",                  2),
        (r"\bbuffer\b",                            1),
        (r"\bstorage\b",                           1),
        (r"\bword[\s\-]?line\b",                   2),
        (r"\bbit[\s\-]?line\b",                    2),
        (r"\bcache\b",                             3),
    ],
 
    # ── Control & Logic Modules ───────────────────────────────────────────────
    # Decoders, MUXes, state machines, arbiters, bus controllers
    # Evidence: "state machine"(14), "decoder"(5), "encoder"(8),
    # "multiplexer"(5), "arbitration"(6), "priority encoder"(2),
    # "clock enable"(23), "enable signal"(22)
    "control_logic": [
        # Named control structures — highest confidence
        (r"\bstate[\s\-]?machine\b",               3),
        (r"\bfinite[\s\-]?state\b",                3),
        (r"\bfsm\b",                               3),
        (r"\bdecoder\b",                           3),
        (r"\bencoder\b",                           3),
        (r"\bpriority[\s\-]?encoder\b",            3),
        (r"\bmultiplexer\b",                       3),
        (r"\bdemultiplexer\b",                     3),
        (r"\bmux\b",                               3),
        (r"\bdemux\b",                             3),
        (r"\barbiter\b",                           3),
        (r"\barbitration\b",                       3),
        (r"\bbus[\s\-]?controller\b",              3),
        (r"\bbus[\s\-]?lock\b",                    3),
        (r"\bbus[\s\-]?arbitrat",                  3),
        (r"\bmealy\b",                             3),
        (r"\bmoore\b",                             3),
        # Flip-flops and latches — sequential control primitives
        (r"\bflip[\s\-]?flop\b",                   3),
        (r"\bd[\s\-]?latch\b",                     3),
        (r"\bd[\s\-]?type[\s\w]*flip[\s\-]?flop\b", 3),
        (r"\bd[\s\-]flip[\s\-]flop\b",            3),
        (r"\basynchronous[\s\w]*reset\b",          2),
        (r"\bsynchronous[\s\w]*reset\b",           2),
        # Control flow and signal routing
        (r"\bcontrol[\s\-]?flow\b",                2),
        (r"\boperational[\s\-]?sequence\b",        2),
        (r"\bselect[\s\-]?signal\b",               2),
        (r"\benable[\s\-]?signal\b",               1),
        (r"\bclock[\s\-]?enable\b",                2),
        (r"\bclock[\s\-]?gating\b",                2),
        (r"\bclock[\s\-]?domain\b",                2),
        (r"\bstate[\s\-]?transition\b",            3),
        (r"\bnext[\s\-]?state\b",                  2),
        (r"\bcurrent[\s\-]?state\b",               2),
        (r"\bpresent[\s\-]?state\b",               2),
        (r"\btruth[\s\-]?table\b",                 2),
        (r"\bcombinational[\s\w]*logic\b",         2),
        (r"\bsequential[\s\w]*logic\b",            2),
        (r"\bcontrol[\s\w]*logic\b",               2),
        (r"\bpriority[\s\w]*logic\b",              2),
        (r"\bpriority[\s\w]*scheme\b",             2),
        (r"\bhandshak",                            2),
        (r"\bready.*valid\b",                      2),
        (r"\bboolean\b",                           1),
        (r"\binverter\b",                          2),
        (r"\bbuffer\b",                            1),  # shared; low weight
        (r"\bgate[\s\w]*logic\b",                  1),
        (r"\bcounter\b",                           2),   # CNT modules observed
        (r"\bcount.*clock\b",                      2),
        (r"\bclock.*count\b",                      2),
    ],
 
    # ── Miscellaneous / Higher-level IP Modules ───────────────────────────────
    # Processor cores, communication protocols, cryptography, video/graphics,
    # AXI bus IP, testbenches, debug utilities
    # Evidence: "axi"(207), "axis"(106), "axi stream"(53), "pixel"(40),
    # "video"(19), "color"(14), "palette"(21), "counter"(40), "timer"(5),
    # "testbench"(3), "simulation"(11), "debugging"(7), "cpu"(9)
    "miscellaneous": [
        # AXI / streaming bus protocols — dominant in dataset
        (r"\baxi[\s\-]?stream\b",                  3),
        (r"\baxi4[\s\-]?stream\b",                 3),
        (r"\baxi[\s\-]?lite\b",                    3),
        (r"\baxis\b",                              2),
        (r"\baxi\b",                               2),
        (r"\bahb\b",                               3),
        (r"\bapb\b",                               3),
        (r"\bwishbone\b",                          3),
        # Other serial/comms protocols
        (r"\buart\b",                              3),
        (r"\bspi\b",                               3),
        (r"\bi2c\b",                               3),
        (r"\bi²c\b",                               3),
        (r"\bpcie\b",                              3),
        (r"\busb\b",                               3),
        (r"\bethernet\b",                          3),
        (r"\bcan[\s\-]?bus\b",                     3),
        (r"\bcommunication[\s\w]*protocol\b",      3),
        (r"\bserial[\s\w]*protocol\b",             2),
        (r"\bprotocol\b",                          1),
        (r"\binterface[\s\w]*standard\b",          2),
        # AXI stream signals (very specific to this protocol family)
        (r"\btdata\b",                             3),
        (r"\btkeep\b",                             3),
        (r"\btlast\b",                             3),
        (r"\btvalid\b",                            2),
        (r"\btready\b",                            2),
        (r"\bframe[\s\w]*join\b",                  3),
        (r"\bframe[\s\w]*length\b",                2),
        (r"\brate[\s\w]*limit",                    2),
        (r"\bcrosspoint\b",                        3),
        (r"\bbroadcast[\s\w]*module\b",            2),
        (r"\bcobs[\s\w]*encod",                    3),
        (r"\bcobs[\s\w]*decod",                    3),
        (r"\bconsistent[\s\w]*overhead\b",         3),
        # Processor / CPU cores
        (r"\bprocessor[\s\w]*core\b",              3),
        (r"\bprocessor\b",                         2),
        (r"\bcpu\b",                               3),
        (r"\brisc\b",                              3),
        (r"\bmips\b",                              3),
        (r"\binstruction[\s\w]*fetch\b",           3),
        (r"\binstruction[\s\w]*decode\b",          3),
        (r"\binstruction[\s\w]*set\b",             3),
        (r"\bpipeline[\s\w]*stage\b",              2),
        (r"\bhazard\b",                            2),
        (r"\bstall\b",                             2),
        # Video / display / graphics
        (r"\bvideo[\s\w]*controller\b",            3),
        (r"\bvideo[\s\w]*timing\b",                3),
        (r"\bcolor[\s\w]*mix",                     3),
        (r"\bpalette\b",                           3),
        (r"\bpixel\b",                             2),
        (r"\bdisplay[\s\w]*buffer\b",              3),
        (r"\bscroll[\s\w]*controller\b",           3),
        (r"\bobject[\s\w]*controller\b",           3),
        (r"\bvga\b",                               3),
        (r"\bhdmi\b",                              3),
        (r"\bvram\b",                              3),
        (r"\bgraphic",                             2),
        (r"\brender",                              2),
        (r"\bblank",                               1),
        (r"\bsync(?:hronization)?[\s\w]*signal\b", 1),
        (r"\bvideo\b",                             2),
        # Cryptography / DSP
        (r"\bcryptograph",                         3),
        (r"\baes\b",                               3),
        (r"\brsa\b",                               3),
        (r"\bdes\b",                               3),
        (r"\bsha\b",                               3),
        (r"\bencrypt",                             3),
        (r"\bdecrypt",                             3),
        (r"\bcipher\b",                            3),
        (r"\bfft\b",                               3),
        (r"\bdft\b",                               3),
        (r"\bfir\b",                               3),
        (r"\biir\b",                               3),
        (r"\bdsp\b",                               3),
        (r"\bdigital[\s\w]*signal[\s\w]*process",  3),
        # Timers / counters (miscellaneous IP)
        (r"\btimer\b",                             2),
        (r"\bwatchdog\b",                          3),
        (r"\bclock[\s\-]?divider\b",               2),
        (r"\bpwm\b",                               3),
        (r"\bpll\b",                               3),
        (r"\bphase[\s\-]?locked\b",                3),
        # System-level IP
        (r"\bdma\b",                               3),
        (r"\bdirect[\s\w]*memory[\s\w]*access\b",  3),
        (r"\binterrupt\b",                         2),
        (r"\bsoc\b",                               3),
        (r"\bsystem[\s\-]?on[\s\-]?chip\b",        3),
        (r"\bembedded[\s\w]*design\b",             2),
        # Testbench / debug / simulation utilities
        (r"\btestbench\b",                         3),
        (r"\btest[\s\-]?bench\b",                  3),
        (r"\bsimulation\b",                        2),
        (r"\bdebugging\b",                         2),
        (r"\bwaveform[\s\w]*dump\b",               3),
        (r"\bvcd\b",                               3),
        (r"\blxt\b",                               2),
        (r"\bstimulus\b",                          2),
        (r"\bdip[\s\-]?switch\b",                  2),
        (r"\bdip[\s\w]*setting\b",                 2),
    ],
}

Pre-compilation of patterns

In [3]:
import re

COMPILED_KEYWORDS: dict[str, list[tuple[re.Pattern, int]]] = {
    cat: [(re.compile(pat, re.IGNORECASE), weight) for pat, weight in kws]
    for cat, kws in CATEGORY_KEYWORDS.items()
}
 
CATEGORY_LABELS = {
    "arithmetic":     "Arithmetic Modules",
    "memory_storage": "Memory & Storage Modules",
    "control_logic":  "Control & Logic Modules",
    "miscellaneous":  "Miscellaneous Modules",
}
 
OUTPUT_FILES = {
    "arithmetic":     "arithmetic_modules.csv",
    "memory_storage": "memory_storage_modules.csv",
    "control_logic":  "control_logic_modules.csv",
    "miscellaneous":  "miscellaneous_modules.csv",
}

Core Functions

In [4]:
import pandas as pd

def score_text(text: str) -> dict[str, int]:
    """
    Score a piece of text against every category's keyword list.
    Returns {category_key: total_weighted_score}.
    Each distinct non-overlapping match contributes its weight once.
    """
    scores = {cat: 0 for cat in COMPILED_KEYWORDS}
    for cat, patterns in COMPILED_KEYWORDS.items():
        for pattern, weight in patterns:
            hits = len(pattern.findall(text))
            scores[cat] += hits * weight
    return scores
 
 
def classify_row(row: pd.Series) -> tuple[str, dict[str, int]]:
    """
    Classify one dataset row.
 
    Scoring strategy
    ----------------
    Primary   : `llm_response`  — rich natural-language descriptions;
                                  full keyword weight applied.
    Secondary : `ioheader`      — module name + port declarations;
                                  applied at half weight as a tiebreaker.
 
    Tie-breaking: highest combined score wins.
    Fallback:     all-zero scores → "miscellaneous".
    """
    llm_text = str(row.get("llm_response", "") or "")
    io_text  = str(row.get("ioheader",     "") or "")
 
    scores_llm = score_text(llm_text)
    scores_io  = score_text(io_text)
 
    combined = {
        cat: scores_llm[cat] + scores_io[cat] // 2
        for cat in COMPILED_KEYWORDS
    }
 
    winner = max(combined, key=combined.get)
    if combined[winner] == 0:
        winner = "miscellaneous"
 
    return winner, combined

Main classification pipeline

In [5]:
import os
import pandas as pd
from tqdm import tqdm

def filter_Save(csv_path: str = CSV_PATH, output_dir: str = OUTPUT_DIR):
 
    # ── 1. Load CSV ───────────────────────────────────────────────────────────
    print(f"Loading CSV: {csv_path} …")
    df = pd.read_csv(csv_path)
    print(f"      {len(df):,} rows  x  {len(df.columns)} columns loaded.")
    print(f"      Columns: {list(df.columns)}\n")
 
    if "llm_response" not in df.columns:
        raise ValueError("Column 'llm_response' not found in the CSV.")
 
    # ── 2. Classify ───────────────────────────────────────────────────────────
    print("Classifying rows …")
 
    categories, cat_scores, all_scores, ambiguous_flags = [], [], [], []
 
    for _, row in tqdm(df.iterrows(), total=len(df), desc="  Scoring", unit="row"):
        category, scores = classify_row(row)
        categories.append(category)
        cat_scores.append(scores[category])
        all_scores.append(scores)
 
        sorted_vals = sorted(scores.values(), reverse=True)
        is_ambiguous = (
            len(sorted_vals) >= 2
            and sorted_vals[0] > 0
            and (sorted_vals[1] / sorted_vals[0]) >= 0.80
        )
        ambiguous_flags.append(is_ambiguous)
 
    # Attach columns
    df["category"]              = [CATEGORY_LABELS[c] for c in categories]
    df["category_key"]          = categories
    df["category_score"]        = cat_scores
    df["score_arithmetic"]      = [s["arithmetic"]     for s in all_scores]
    df["score_memory_storage"]  = [s["memory_storage"] for s in all_scores]
    df["score_control_logic"]   = [s["control_logic"]  for s in all_scores]
    df["score_miscellaneous"]   = [s["miscellaneous"]  for s in all_scores]
    df["is_ambiguous"]          = ambiguous_flags
    df["fault_type"]            = "no_fault"
    df["fault_label"]           = 0
    df["fault_site"]            = None
 
    # ── 3. Report ─────────────────────────────────────────────────────────────
    print(f"\nClassification summary")
    print(f"{'─'*50}")
    total = len(df)
    counts = df["category_key"].value_counts()
    for key, label in CATEGORY_LABELS.items():
        n = counts.get(key, 0)
        print(f"  {label:<32s}: {n:4d}  ({100 * n / total:.1f}%)")
    print(f"  {'─'*48}")
    print(f"  {'Total':<32s}: {total:4d}")
    print(f"  Ambiguous (top-2 within 20 %)  : {sum(ambiguous_flags):4d}\n")
 
    # ── 4. Save CSVs ──────────────────────────────────────────────────────────
    print(f"Saving category CSVs to '{output_dir}/' …")
    os.makedirs(output_dir, exist_ok=True)
 
    for key, fname in OUTPUT_FILES.items():
        subset = df[df["category_key"] == key].drop(columns=["category_key"])
        path   = os.path.join(output_dir, fname)
        subset.to_csv(path, index=False)
        print(f"  {fname:<45s}  ({len(subset):4d} rows)")
 
    print(f"\n  All files written to: {output_dir}/")
    return df

if os.path.exists(CSV_PATH):
    filter_Save(CSV_PATH, OUTPUT_DIR)

Loading CSV: escad_openrtlset.csv …
      131,309 rows  x  8 columns loaded.
      Columns: ['index', 'ioheader', 'verilog_code', 'full_code', 'llm_response', 'Repo_url', 'lic_name', 'children_index']

Classifying rows …


  Scoring: 100%|██████████| 131309/131309 [11:29<00:00, 190.39row/s]



Classification summary
──────────────────────────────────────────────────
  Arithmetic Modules              : 27521  (21.0%)
  Memory & Storage Modules        : 18395  (14.0%)
  Control & Logic Modules         : 60586  (46.1%)
  Miscellaneous Modules           : 24807  (18.9%)
  ────────────────────────────────────────────────
  Total                           : 131309
  Ambiguous (top-2 within 20 %)  : 11800

Saving category CSVs to 'escad_openrtlset_categorized/' …
  arithmetic_modules.csv                         (27521 rows)
  memory_storage_modules.csv                     (18395 rows)
  control_logic_modules.csv                      (60586 rows)
  miscellaneous_modules.csv                      (24807 rows)

  All files written to: escad_openrtlset_categorized/


---
### 3. Constants and Configuration

Import the required dependencies

In [6]:
import re
import os
import math
import warnings
import pandas as pd
import numpy as np
import networkx as nx
from collections import defaultdict
from typing import Dict, List, Tuple, Optional
 
import torch
from torch_geometric.data import Data, InMemoryDataset
from torch_geometric.utils import from_networkx
from sklearn.preprocessing import LabelEncoder
 
warnings.filterwarnings("ignore")

Configurations

In [7]:
CSV_DIR    = "escad_openrtlset_categorized"
N_SAMPLES  = 1000
SAVE_DIR   = "VLSI_PYG_Set"

CSV_PATHS = [os.path.join(CSV_DIR, f) for f in os.listdir(CSV_DIR) if f.endswith(".csv") and not f.startswith("._")]

KNOWN_GATES = [
    "AND", "OR", "XOR", "NOT", "NAND", "NOR", "XNOR", "BUF",
    "MUX", "DFF", "LATCH", "DFFR", "DFFS", "INPUT", "OUTPUT", "OTHER"
]
GATE_TO_IDX = {g: i for i, g in enumerate(KNOWN_GATES)}
 
FAULT_LABEL_MAP = {
    "no_fault":       0,
    "none":           0,
    "stuck_at_0":     1,
    "sa0":            1,
    "stuck_at_1":     2,
    "sa1":            2,
    "delay":          3,
    "delay_fault":    3,
    "bridging":       4,
    "bridging_fault": 4,
    "open":           5,
    "open_fault":     5,
}

---
### 4. Verilog Parser

Lightweight structural Verilog parser that extracts:
- module ports (inputs / outputs)
- wire declarations and their widths
- gate / cell instantiations with pin connections

In [8]:
class VerilogParser:
     
    # Matches: gate_type  [optional_strength_or_drive]  instance_name ( port_list );
    GATE_RE = re.compile(
        r'\b(and|or|xor|not|nand|nor|xnor|buf|mux|dff\w*|latch\w*)\s+'
        r'(?:#\([^)]*\)\s*)?(\w+)\s*\(([^;]+)\)\s*;',
        re.IGNORECASE | re.DOTALL
    )
    # Matches: module_name  instance_name ( .port(signal), ... );
    CELL_RE = re.compile(
        r'(\b\w+\b)\s+(\w+)\s*\(([^;]+)\)\s*;',
        re.IGNORECASE | re.DOTALL
    )
    INPUT_RE  = re.compile(r'\binput\b\s+(?:\[(\d+):(\d+)\]\s+)?([^;]+);', re.IGNORECASE)
    OUTPUT_RE = re.compile(r'\boutput\b\s+(?:\[(\d+):(\d+)\]\s+)?([^;]+);', re.IGNORECASE)
    WIRE_RE   = re.compile(r'\bwire\b\s+(?:\[(\d+):(\d+)\]\s+)?([^;]+);', re.IGNORECASE)
 
    def parse(self, verilog_code: str) -> nx.DiGraph:
        """Return a directed graph where nodes are signals/gates."""
        G = nx.DiGraph()
 
        inputs  = self._parse_ports(verilog_code, self.INPUT_RE,  "PI")
        outputs = self._parse_ports(verilog_code, self.OUTPUT_RE, "PO")
        wires   = self._parse_ports(verilog_code, self.WIRE_RE,   "WIRE")
 
        all_signals = {**inputs, **outputs, **wires}
 
        # Add signal nodes
        for sig, info in all_signals.items():
            G.add_node(sig, **info)
 
        # Add gate instances
        gate_instances = self._parse_gates(verilog_code)
        for inst in gate_instances:
            gate_node = inst["name"]
            G.add_node(
                gate_node,
                gate_type  = inst["gate_type"],
                is_PI      = False,
                is_PO      = False,
                is_seq     = inst["is_seq"],
                bus_width  = 1,
                node_type  = "GATE"
            )
            # Inputs → gate
            for sig in inst["inputs"]:
                if sig:
                    if sig not in G:
                        G.add_node(sig, gate_type="INPUT", is_PI=False,
                                   is_PO=False, is_seq=False, bus_width=1, node_type="WIRE")
                    G.add_edge(sig, gate_node)
            # Gate → outputs
            for sig in inst["outputs"]:
                if sig:
                    if sig not in G:
                        G.add_node(sig, gate_type="OUTPUT", is_PI=False,
                                   is_PO=sig in outputs, is_seq=False,
                                   bus_width=all_signals.get(sig, {}).get("bus_width", 1),
                                   node_type="WIRE")
                    G.add_edge(gate_node, sig)
 
        # Mark PI / PO
        for sig in inputs:
            if sig in G:
                G.nodes[sig]["is_PI"] = True
        for sig in outputs:
            if sig in G:
                G.nodes[sig]["is_PO"] = True
 
        return G if G.number_of_nodes() > 0 else self._fallback_graph()
 
    def _parse_ports(self, code: str, pattern, node_type: str) -> Dict:
        result = {}
        for m in pattern.finditer(code):
            hi_str, lo_str = m.group(1), m.group(2)
            width = (int(hi_str) - int(lo_str) + 1) if hi_str else 1
            for sig in m.group(3).split(","):
                sig = sig.strip()
                if sig:
                    result[sig] = dict(
                        gate_type = node_type,
                        is_PI     = node_type == "PI",
                        is_PO     = node_type == "PO",
                        is_seq    = False,
                        bus_width = width,
                        node_type = node_type,
                    )
        return result
 
    def _parse_gates(self, code: str) -> List[Dict]:
        instances = []
        for m in self.GATE_RE.finditer(code):
            gate_type = m.group(1).upper()
            inst_name = m.group(2)
            port_str  = m.group(3)
            ports     = [p.strip() for p in port_str.split(",") if p.strip()]
 
            is_seq = gate_type in ("DFF", "LATCH") or gate_type.startswith("DFF")
 
            # Heuristic: first port = output, rest = inputs
            instances.append(dict(
                gate_type = gate_type,
                name      = inst_name,
                outputs   = [ports[0]] if ports else [],
                inputs    = ports[1:]  if len(ports) > 1 else [],
                is_seq    = is_seq,
            ))
        return instances
 
    @staticmethod
    def _fallback_graph() -> nx.DiGraph:
        """Return a minimal valid graph when parsing yields nothing."""
        G = nx.DiGraph()
        G.add_node("stub", gate_type="OTHER", is_PI=False, is_PO=False,
                   is_seq=False, bus_width=1, node_type="WIRE")
        return G


---
### 5. Feature Extractor

Computes per-node features from a circuit graph:
- [0..15]  `gate_type` one-hot  (16 types)
- [16]     `fan_in`
- [17]     `fan_out`
- [18]     `logic_depth`  (BFS distance from any PI)
- [19]     `is_PI`
- [20]     `is_PO`
- [21]     `is_seq`  (flip-flop / latch flag)
- [22]     `bus_width`
- [23]     `in_degree`
- [24]     `out_degree`
- [25]     `CC0`  (combinational controllability to 0, normalised)
- [26]     `CC1`  (combinational controllability to 1, normalised)
- [27]     `CO`   (combinational observability, normalised)
- [28]     `gate_level`           (topological level)
- [29]     `num_downstream_gates`
- [30]     `dist_to_nearest_output`
- [31]     `dist_to_nearest_input`
- [32]     `reconvergent_fanout_flag`
- [33]     `critical_path_membership`
- [34]     `clock_domain_id`      (0 if combinational, 1+ for sequential domains)
 
Total: 35 features per node.

In [9]:
class FeatureExtractor:
 
    FEATURE_DIM = 35
 
    def extract(self, G: nx.DiGraph) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        """
        Returns
        -------
        x          : FloatTensor [N, 35]
        edge_index : LongTensor  [2, E]
        node_order : list of node names (same order as rows in x)
        """
        nodes = list(G.nodes())
        N     = len(nodes)
        idx   = {n: i for i, n in enumerate(nodes)}
 
        # --- Pre-compute graph-level metrics ---
        logic_depth          = self._logic_depth(G, nodes)
        gate_level           = self._gate_level(G, nodes)
        num_downstream       = self._downstream_count(G, nodes)
        dist_to_output       = self._dist_to_nearest(G, nodes, target="PO")
        dist_to_input        = self._dist_to_nearest(G, nodes, target="PI")
        reconvergent         = self._reconvergent_flags(G, nodes)
        critical_path_set    = self._critical_path(G, nodes)
        clock_domain         = self._clock_domains(G, nodes)
        cc0, cc1, co         = self._scoap(G, nodes)
 
        # Normalisation helpers
        def norm(val, max_val):
            return float(val) / (float(max_val) + 1e-9)
 
        max_depth     = max(logic_depth.values())  or 1
        max_level     = max(gate_level.values())   or 1
        max_downstream= max(num_downstream.values()) or 1
        max_dist_out  = max(dist_to_output.values()) or 1
        max_dist_in   = max(dist_to_input.values())  or 1
 
        x = np.zeros((N, self.FEATURE_DIM), dtype=np.float32)
 
        for i, node in enumerate(nodes):
            attr = G.nodes[node]
 
            # --- Gate type one-hot [0..15] ---
            raw_type = str(attr.get("gate_type", "OTHER")).upper()
            # Collapse DFF variants
            if raw_type.startswith("DFF") or raw_type.startswith("LATCH"):
                raw_type = "DFF"
            gate_idx = GATE_TO_IDX.get(raw_type, GATE_TO_IDX["OTHER"])
            x[i, gate_idx] = 1.0
 
            # --- Fan-in / Fan-out [16, 17] ---
            x[i, 16] = norm(G.in_degree(node),  N)
            x[i, 17] = norm(G.out_degree(node), N)
 
            # --- Logic depth [18] ---
            x[i, 18] = norm(logic_depth[node], max_depth)
 
            # --- PI / PO / Sequential flags [19, 20, 21] ---
            x[i, 19] = float(bool(attr.get("is_PI", False)))
            x[i, 20] = float(bool(attr.get("is_PO", False)))
            x[i, 21] = float(bool(attr.get("is_seq", False)))
 
            # --- Bus width [22] ---
            x[i, 22] = norm(attr.get("bus_width", 1), 64)
 
            # --- In-degree / Out-degree [23, 24] ---
            x[i, 23] = float(G.in_degree(node))
            x[i, 24] = float(G.out_degree(node))
 
            # --- SCOAP-inspired [25, 26, 27] ---
            x[i, 25] = norm(cc0[node], N * 10)
            x[i, 26] = norm(cc1[node], N * 10)
            x[i, 27] = norm(co[node],  N * 10)
 
            # --- Extended features [28..34] ---
            x[i, 28] = norm(gate_level[node],          max_level)
            x[i, 29] = norm(num_downstream[node],      max_downstream)
            x[i, 30] = norm(dist_to_output[node],      max_dist_out)
            x[i, 31] = norm(dist_to_input[node],       max_dist_in)
            x[i, 32] = float(reconvergent[node])
            x[i, 33] = float(node in critical_path_set)
            x[i, 34] = float(clock_domain.get(node, 0))
 
        # --- Edge index ---
        src, dst = [], []
        for u, v in G.edges():
            if u in idx and v in idx:
                src.append(idx[u])
                dst.append(idx[v])
 
        edge_index = torch.tensor([src, dst], dtype=torch.long)
        return torch.tensor(x, dtype=torch.float), edge_index, nodes
 
    # ------------------------------------------------------------------
    # Internal helpers
    # ------------------------------------------------------------------
 
    def _logic_depth(self, G, nodes) -> Dict:
        """BFS from PI nodes; depth = longest shortest path from any PI."""
        depth = {n: 0 for n in nodes}
        pi_nodes = [n for n in nodes if G.nodes[n].get("is_PI", False)]
        if not pi_nodes:
            pi_nodes = [n for n in nodes if G.in_degree(n) == 0]
        for pi in pi_nodes:
            lengths = nx.single_source_shortest_path_length(G, pi)
            for node, d in lengths.items():
                depth[node] = max(depth.get(node, 0), d)
        return depth
 
    def _gate_level(self, G, nodes) -> Dict:
        """Topological level (longest path from any source)."""
        level = {n: 0 for n in nodes}
        try:
            for node in nx.topological_sort(G):
                for succ in G.successors(node):
                    level[succ] = max(level[succ], level[node] + 1)
        except nx.NetworkXUnfeasible:
            pass
        return level
 
    def _downstream_count(self, G, nodes) -> Dict:
        """Number of nodes reachable downstream from each node."""
        count = {}
        for node in nodes:
            count[node] = len(nx.descendants(G, node))
        return count
 
    def _dist_to_nearest(self, G, nodes, target: str) -> Dict:
        """Shortest path distance to nearest PI or PO."""
        target_nodes = [n for n in nodes if G.nodes[n].get(
            "is_PI" if target == "PI" else "is_PO", False)]
        dist = {n: float("inf") for n in nodes}
 
        if target == "PO":
            # forward reachability from node to PO
            R = G
        else:
            # reverse reachability from node back to PI
            R = G.reverse(copy=False)
 
        for t in target_nodes:
            lengths = nx.single_source_shortest_path_length(R, t)
            for node, d in lengths.items():
                dist[node] = min(dist.get(node, float("inf")), d)
 
        # Replace inf with a large sentinel
        max_finite = max((v for v in dist.values() if v != float("inf")), default=0)
        sentinel   = max_finite + 1
        return {n: (v if v != float("inf") else sentinel) for n, v in dist.items()}
 
    def _reconvergent_flags(self, G, nodes) -> Dict:
        """
        A node is reconvergent fan-out if it has multiple paths to the
        same downstream node (i.e., multiple successors whose ancestor
        sets overlap at some common descendant).
        Simple approximation: flag nodes with out_degree > 1 where any
        pair of successors share a common descendant.
        """
        flags = {n: False for n in nodes}
        for node in nodes:
            succs = list(G.successors(node))
            if len(succs) < 2:
                continue
            desc_sets = [nx.descendants(G, s) | {s} for s in succs]
            # Check pairwise intersection
            for a in range(len(desc_sets)):
                for b in range(a + 1, len(desc_sets)):
                    if desc_sets[a] & desc_sets[b]:
                        flags[node] = True
                        break
                if flags[node]:
                    break
        return flags
 
    def _critical_path(self, G, nodes) -> set:
        """Nodes on the longest topological path (proxy for timing-critical)."""
        level = self._gate_level(G, nodes)
        if not level:
            return set()
        max_level = max(level.values())
        # Walk backwards from max-level PO nodes
        critical = set()
        try:
            for node in reversed(list(nx.topological_sort(G))):
                if level[node] == max_level or node in critical:
                    critical.add(node)
                    for pred in G.predecessors(node):
                        if level.get(pred, 0) == level[node] - 1:
                            critical.add(pred)
        except nx.NetworkXUnfeasible:
            pass
        return critical
 
    def _clock_domains(self, G, nodes) -> Dict:
        """
        Assign clock domain IDs by BFS from sequential (DFF/latch) nodes.
        Combinational nodes inherit the domain of their driving sequential element.
        """
        domain = {n: 0 for n in nodes}
        seq_nodes = [n for n in nodes if G.nodes[n].get("is_seq", False)]
        for d_id, seq in enumerate(seq_nodes, start=1):
            domain[seq] = d_id
            for succ in nx.descendants(G, seq):
                if domain.get(succ, 0) == 0:
                    domain[succ] = d_id
        return domain
 
    def _scoap(self, G, nodes) -> Tuple[Dict, Dict, Dict]:
        """
        Approximate SCOAP controllability / observability scores.
        CC0(PI) = CC1(PI) = 1;  propagated through gate types.
        CO(PO)  = 0;             propagated backwards.
        """
        cc0 = {n: 9999 for n in nodes}
        cc1 = {n: 9999 for n in nodes}
        co  = {n: 9999 for n in nodes}
 
        # Initialise PIs
        for n in nodes:
            if G.nodes[n].get("is_PI", False):
                cc0[n] = 1
                cc1[n] = 1
 
        # Forward pass (topological order)
        try:
            topo = list(nx.topological_sort(G))
        except nx.NetworkXUnfeasible:
            topo = nodes
 
        for node in topo:
            preds = list(G.predecessors(node))
            if not preds:
                continue
            gate = str(G.nodes[node].get("gate_type", "OTHER")).upper()
            p_cc0 = [cc0[p] for p in preds]
            p_cc1 = [cc1[p] for p in preds]
 
            if gate in ("AND", "NAND"):
                cc1[node] = 1 + sum(p_cc1)
                cc0[node] = 1 + min(p_cc0)
                if gate == "NAND":
                    cc0[node], cc1[node] = cc1[node], cc0[node]
            elif gate in ("OR", "NOR"):
                cc0[node] = 1 + sum(p_cc0)
                cc1[node] = 1 + min(p_cc1)
                if gate == "NOR":
                    cc0[node], cc1[node] = cc1[node], cc0[node]
            elif gate in ("XOR", "XNOR"):
                cc1[node] = 1 + min(p_cc1[0] + p_cc0[1] if len(p_cc0) > 1 else p_cc1[0],
                                    p_cc0[0] + p_cc1[1] if len(p_cc1) > 1 else p_cc0[0])
                cc0[node] = 1 + min(p_cc0[0] + p_cc0[1] if len(p_cc0) > 1 else p_cc0[0],
                                    p_cc1[0] + p_cc1[1] if len(p_cc1) > 1 else p_cc1[0])
                if gate == "XNOR":
                    cc0[node], cc1[node] = cc1[node], cc0[node]
            elif gate in ("NOT", "BUF"):
                cc0[node] = 1 + (p_cc1[0] if gate == "NOT" else p_cc0[0])
                cc1[node] = 1 + (p_cc0[0] if gate == "NOT" else p_cc1[0])
            elif gate in ("DFF", "LATCH"):
                # Sequential: pass through with extra cost
                cc0[node] = 1 + (min(p_cc0) if p_cc0 else 9999)
                cc1[node] = 1 + (min(p_cc1) if p_cc1 else 9999)
            else:
                cc0[node] = 1 + (min(p_cc0) if p_cc0 else 9999)
                cc1[node] = 1 + (min(p_cc1) if p_cc1 else 9999)
 
        # Observability: backward pass
        for n in nodes:
            if G.nodes[n].get("is_PO", False):
                co[n] = 0
 
        for node in reversed(topo):
            succs = list(G.successors(node))
            if not succs:
                continue
            co[node] = 1 + min(co[s] for s in succs)
 
        return cc0, cc1, co


---
### 6. PyTorch Geometric Dataset

PyTorch Geometric InMemoryDataset for VLSI fault detection.

Each sample from the CSV becomes one `Data` object with:
- `data.x`          - node features  [N, 35]
- `data.edge_index` - directed edges [2, E]
- `data.y`          - graph-level fault label (`int64` scalar)
- `data.num_nodes`  - N

In [10]:
class VLSIFaultDataset(InMemoryDataset):
 
    def __init__(self, root: str, data_list=None, transform=None):
        self._data_list_input = data_list
        super().__init__(root, transform)
        if self._data_list_input is not None:
            os.makedirs(self.processed_dir, exist_ok=True)
            self.process()
        self.data, self.slices = torch.load(self.processed_paths[0])

    @property
    def raw_file_names(self):
        return []
 
    @property
    def processed_file_names(self):
        return ["vlsi_fault_data.pt"]
 
    def download(self):
        pass
 
    def process(self):
        torch.save(
            self.collate(self._data_list_input),
            self.processed_paths[0]
        )

Main Pipeline

In [11]:
import concurrent.futures
import gc

def infer_fault_label(row: pd.Series) -> int:
    """
    Try several common column names to find the fault type.
    Falls back to 0 (no fault) if nothing is found.
    """
    candidate_cols = [
        "fault_type", "fault", "label", "class",
        "fault_label", "fault_category", "error_type"
    ]
    for col in candidate_cols:
        if col in row.index:
            val = str(row[col]).strip().lower().replace(" ", "_").replace("-", "_")
            if val in FAULT_LABEL_MAP:
                return FAULT_LABEL_MAP[val]
            # Try prefix matching
            for key, mapped in FAULT_LABEL_MAP.items():
                if val.startswith(key) or key.startswith(val):
                    return mapped
    return 0   # default: no fault


def find_verilog_column(df: pd.DataFrame) -> Optional[str]:
    """Detect which column holds raw Verilog code."""
    candidates = [
        "verilog", "rtl", "code", "verilog_code",
        "rtl_code", "design", "source", "module"
    ]
    for col in candidates:
        if col in df.columns:
            return col
    # Fallback: pick the column with the longest average string
    str_cols = df.select_dtypes(include="object").columns.tolist()
    if str_cols:
        avg_len = {c: df[c].dropna().apply(len).mean() for c in str_cols}
        return max(avg_len, key=avg_len.get)
    return None


def build_dataset(csv_path: str, n_samples: int, sample_start: int, save_dir: str = SAVE_DIR):
    print(f"{'─'*50}")
    print("NeuroChip SmartSilicon: GNN Dataset Builder")
    print(f"{'─'*50}")

    if sample_start < 0:
        raise ValueError("sample_start must be >= 0")

    # ── Load CSV ──────────────────────────────────────────────────────
    print(f"Loading CSV: {csv_path}")
    print(f"      Using rows {sample_start}..{sample_start + n_samples - 1}")
    df = pd.read_csv(csv_path, skiprows=range(1, sample_start + 1), nrows=n_samples)
    df = df.reset_index(drop=True)
    print(f"      Loaded {len(df)} rows, {len(df.columns)} columns")
    print(f"      Columns: {list(df.columns)}\n")

    verilog_col = find_verilog_column(df)
    if verilog_col is None:
        raise ValueError("Could not detect a Verilog code column in the CSV.")
    print(f"Verilog column detected: '{verilog_col}'\n")

    # Worker: parse verilog and extract features -> Data object
    def _parse_extract_pair(verilog_code: str, fault_label: int):
        try:
            parser = VerilogParser()
            extractor = FeatureExtractor()
            G = parser.parse(verilog_code)
            x, edge_index, _ = extractor.extract(G)
        except Exception:
            # Fallback to a 1-node stub
            x = torch.zeros(1, FeatureExtractor.FEATURE_DIM)
            edge_index = torch.zeros(2, 0, dtype=torch.long)
        data = Data(
            x=x,
            edge_index=edge_index,
            y=torch.tensor([fault_label], dtype=torch.long),
            num_nodes=x.size(0)
        )
        # free large temporary objects if present
        try:
            del G
        except Exception:
            pass
        return data

    # ── Parse & extract features in parallel (threaded for safety in notebook)
    print("Parsing Verilog and extracting features (parallel) …")
    data_list = []
    skipped = 0
    label_counts = defaultdict(int)

    max_workers = min(8, (os.cpu_count() or 1))
    with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as ex:
        futures = []
        for row in df.itertuples(index=False):
            # Access column by name dynamically
            verilog_code = getattr(row, verilog_col) if hasattr(row, verilog_col) else str(row[0])
            fault_label = infer_fault_label(pd.Series(row._asdict())) if hasattr(row, '_asdict') else 0
            label_counts[fault_label] += 1
            futures.append(ex.submit(_parse_extract_pair, verilog_code, fault_label))

        # Collect results as they complete to avoid peak memory spikes
        for i, fut in enumerate(concurrent.futures.as_completed(futures), start=1):
            try:
                data = fut.result()
                data_list.append(data)
            except Exception as e:
                skipped += 1
                print(f"  [WARN] Parsing task failed: {e}")

            if i % 50 == 0:
                print(f"  … processed {i}/{len(futures)} samples")

    # clear large temporary data structures
    try:
        del futures
        del ex
        gc.collect()
    except Exception:
        pass

    print(f"\n  Done - {len(data_list)} graphs built, {skipped} used stub fallback.")
    print(f"\n  Fault-label distribution:")
    label_names = {
        0: "No fault", 1: "Stuck-at-0", 2: "Stuck-at-1",
        3: "Delay fault", 4: "Bridging fault", 5: "Open fault"
    }
    for lbl, cnt in sorted(label_counts.items()):
        print(f"    [{lbl}] {label_names[lbl]:<20s}: {cnt:4d} samples")

    # ── Summary ───────────────────────────────────────────────────────
    print(f"{'─'*50}")
    print(f"\nDataset summary for '{csv_path}':")
    sample = data_list[0]
    print(f"  Sample[0].x.shape         : {sample.x.shape}")
    print(f"  Sample[0].edge_index.shape: {sample.edge_index.shape}")
    print(f"  Sample[0].y               : {sample.y}")
    print(f"  Sample[0].num_nodes       : {sample.num_nodes}")
    print(f"\n  Feature vector layout (35 dims):")
    print(f"    [0:15] gate_type one-hot ({len(KNOWN_GATES)} types)")
    print(f"    [16]    fan_in (normalised)")
    print(f"    [17]    fan_out (normalised)")
    print(f"    [18]    logic_depth (normalised)")
    print(f"    [19]    is_PI flag")
    print(f"    [20]    is_PO flag")
    print(f"    [21]    is_sequential flag")
    print(f"    [22]    bus_width (normalised)")
    print(f"    [23]    in_degree (raw)")
    print(f"    [24]    out_degree (raw)")
    print(f"    [25]    CC0 - controllability to 0 (SCOAP, normalised)")
    print(f"    [26]    CC1 - controllability to 1 (SCOAP, normalised)")
    print(f"    [27]    CO  - observability (SCOAP, normalised)")
    print(f"    [28]    gate_level (topological, normalised)")
    print(f"    [29]    num_downstream_gates (normalised)")
    print(f"    [30]    dist_to_nearest_output (normalised)")
    print(f"    [31]    dist_to_nearest_input (normalised)")
    print(f"    [32]    reconvergent_fanout_flag")
    print(f"    [33]    critical_path_membership")
    print(f"    [34]    clock_domain_id (normalised)")
    print("\nDataset processing completed for GNN training.\n")

    return data_list

---
### 7. Fault Injection Pipeline

Prepare faulty circuits by manually injecting common design faults in them, followed by the creation of a faulty PyG dataset.

Configuration and Imports

In [12]:
import re
import os
import random
import warnings
import pandas as pd
from typing import Optional

warnings.filterwarnings("ignore")

INPUT_DIR  = "escad_openrtlset_categorized"   # source CSVs (clean circuits)
OUTPUT_DIR = "escad_openrtlset_faulty"        # destination for faulty CSVs
RANDOM_SEED = 42

CATEGORY_FILES = {
    "arithmetic":     "arithmetic_modules.csv",
    "control_logic":  "control_logic_modules.csv",
    "memory_storage": "memory_storage_modules.csv",
    "miscellaneous":  "miscellaneous_modules.csv",
}

FAULT_LABEL_MAP = {
    "no_fault":       0,
    "stuck_at_0":     1,
    "stuck_at_1":     2,
    "delay_fault":    3,
    "bridging_fault": 4,
    "open_fault":     5
}

FAULT_NAMES = {v: k for k, v in FAULT_LABEL_MAP.items()}   # int → str

# Regexes reused from VerilogParser in the original notebook
_INPUT_RE  = re.compile(r'\binput\b\s+(?:\[\d+:\d+\]\s+)?([^;]+);',  re.IGNORECASE)
_OUTPUT_RE = re.compile(r'\boutput\b\s+(?:\[\d+:\d+\]\s+)?([^;]+);', re.IGNORECASE)
_WIRE_RE   = re.compile(r'\bwire\b\s+(?:\[\d+:\d+\]\s+)?([^;]+);',   re.IGNORECASE)
_REG_RE    = re.compile(r'\breg\b\s+(?:\[\d+:\d+\]\s+)?([^;]+);',    re.IGNORECASE)

Helper Functions

In [13]:
# Return a deduplicated list of signal names from a Verilog module
def _extract_signals(verilog_code: str) -> list[str]:
    signals: list[str] = []
    for pattern in (_INPUT_RE, _OUTPUT_RE, _WIRE_RE, _REG_RE):
        for match in pattern.finditer(verilog_code):
            for sig in match.group(1).split(","):
                sig = sig.strip().split("[")[0].strip()   # strip bus ranges
                if sig and re.match(r'^\w+$', sig):
                    signals.append(sig)
    return list(dict.fromkeys(signals))

# Stuck-at-0: force *site* permanently to logic 0 (Appends `assign <site> = 1'b0;` inside the module body)
# Any existing continuous assignment to the same wire is commented out.
def _inject_stuck_at_0(verilog: str, site: str) -> str:
    # Comment out existing assignments to the site to avoid multi-driver errors
    verilog = re.sub(
        rf'(\bassign\b\s+{re.escape(site)}\b\s*=\s*[^;]+;)',
        r'// [SA0-FAULT] \1',
        verilog,
        flags=re.IGNORECASE,
    )
    # Insert force assignment just before endmodule
    inject = f"\n    assign {site} = 1'b0; // [SA0-FAULT] stuck-at-0 on {site}\n"
    verilog = re.sub(r'(\bendmodule\b)', inject + r'\1', verilog, flags=re.IGNORECASE)
    return verilog

# Stuck-at-1: force *site* permanently to logic 1
def _inject_stuck_at_1(verilog: str, site: str) -> str:
    verilog = re.sub(
        rf'(\bassign\b\s+{re.escape(site)}\b\s*=\s*[^;]+;)',
        r'// [SA1-FAULT] \1',
        verilog,
        flags=re.IGNORECASE,
    )
    inject = f"\n    assign {site} = 1'b1; // [SA1-FAULT] stuck-at-1 on {site}\n"
    verilog = re.sub(r'(\bendmodule\b)', inject + r'\1', verilog, flags=re.IGNORECASE)
    return verilog

# Delay fault: insert a transport delay (`#1`) on every assignment that drives *site*.  If no assignment exists a dummy `#1` buffer is appended.
# The `#N` construct is a standard Verilog simulation delay.
def _inject_delay_fault(verilog: str, site: str) -> str:
    delay_re = re.compile(
        rf'(\bassign\b\s+{re.escape(site)}\b\s*=\s*)([^;]+;)',
        re.IGNORECASE,
    )
    if delay_re.search(verilog):
        verilog = delay_re.sub(r'\1 #1 \2 // [DELAY-FAULT]', verilog)
    else:
        inject = (
            f"\n    wire {site}_delay_tap;"
            f"\n    assign #1 {site}_delay_tap = {site}; "
            f"// [DELAY-FAULT] artificial delay on {site}\n"
        )
        verilog = re.sub(r'(\bendmodule\b)', inject + r'\1', verilog, flags=re.IGNORECASE)
    return verilog

# Bridging fault: short *site* to a neighbouring signal (the first signal in the list that is different from *site*).  Modelled as `assign <site> = <neighbour> | <site>;`  — i.e. the two nets are wired together via an unintended connection (OR-bridge).
def _inject_bridging_fault(verilog: str, site: str, signals: list[str]) -> str:
    neighbour = next((s for s in signals if s != site), None)
    if neighbour is None:
        # Only one signal — inject a self-tie (degenerate but non-crashing)
        neighbour = site

    inject = (
        f"\n    // [BRIDGE-FAULT] short between {site} and {neighbour}"
        f"\n    wire {site}_bridge_shadow = {site} | {neighbour};"
        f"\n    assign {site} = {site}_bridge_shadow;\n"
    )
    verilog = re.sub(r'(\bendmodule\b)', inject + r'\1', verilog, flags=re.IGNORECASE)
    return verilog

# Open fault: disconnect *site* from its driver by commenting out its assignment and floating the wire (assigned to 1'bz, high-impedance).  A floating / undriven net models the physical open-circuit defect.
def _inject_open_fault(verilog: str, site: str) -> str:
    verilog = re.sub(
        rf'(\bassign\b\s+{re.escape(site)}\b\s*=\s*[^;]+;)',
        r'// [OPEN-FAULT] \1',
        verilog,
        flags=re.IGNORECASE,
    )
    inject = (
        f"\n    assign {site} = 1'bz; "
        f"// [OPEN-FAULT] floating net — open circuit on {site}\n"
    )
    verilog = re.sub(r'(\bendmodule\b)', inject + r'\1', verilog, flags=re.IGNORECASE)
    return verilog


# ─────────────────────────────────────────────────────────────────────────────
# Top-level fault injector
# ─────────────────────────────────────────────────────────────────────────────

def inject_faults(verilog: str, rng: random.Random) -> list[dict]:
    """
    Given a clean Verilog string, produce 5 faulty variants — one per fault
    type — each injected at a randomly chosen signal site.

    Parameters
    ----------
    verilog : str
        Original (clean) Verilog source code.
    rng : random.Random
        Seeded RNG for reproducible site selection.

    Returns
    -------
    list of dict, one entry per fault type:
        {
            "verilog_code": <faulty Verilog string>,
            "fault_type":   <canonical string key>,
            "fault_label":  <int 1–5>,
            "fault_site":   <signal name where fault was injected>,
        }
    """
    signals = _extract_signals(verilog)
    results: list[dict] = []

    if not signals:
        # Fallback: inject a syntactic comment — still produces a row per fault
        for label in range(1, 6):
            results.append({
                "verilog_code": verilog + f"\n// [{FAULT_NAMES[label].upper()}] no injectable site found",
                "fault_type":   FAULT_NAMES[label],
                "fault_label":  label,
                "fault_site":   "N/A"
            })
        return results

    # ── 1. Stuck-at-0 ────────────────────────────────────────────────────────
    site = rng.choice(signals)
    results.append({
        "verilog_code": _inject_stuck_at_0(verilog, site),
        "fault_type":   "stuck_at_0",
        "fault_label":  1,
        "fault_site":   site,
    })

    # ── 2. Stuck-at-1 ────────────────────────────────────────────────────────
    site = rng.choice(signals)
    results.append({
        "verilog_code": _inject_stuck_at_1(verilog, site),
        "fault_type":   "stuck_at_1",
        "fault_label":  2,
        "fault_site":   site,
    })

    # ── 3. Delay fault ───────────────────────────────────────────────────────
    site = rng.choice(signals)
    results.append({
        "verilog_code": _inject_delay_fault(verilog, site),
        "fault_type":   "delay_fault",
        "fault_label":  3,
        "fault_site":   site,
    })

    # ── 4. Bridging fault ────────────────────────────────────────────────────
    site = rng.choice(signals)
    results.append({
        "verilog_code": _inject_bridging_fault(verilog, site, signals),
        "fault_type":   "bridging_fault",
        "fault_label":  4,
        "fault_site":   site,
    })

    # ── 5. Open fault ────────────────────────────────────────────────────────
    site = rng.choice(signals)
    results.append({
        "verilog_code": _inject_open_fault(verilog, site),
        "fault_type":   "open_fault",
        "fault_label":  5,
        "fault_site":   site,
    })

    return results


# ─────────────────────────────────────────────────────────────────────────────
# CSV generation — one faulty CSV per category
# ─────────────────────────────────────────────────────────────────────────────

def _find_verilog_column(df: pd.DataFrame) -> Optional[str]:
    """Mirror of find_verilog_column() from the original notebook."""
    candidates = ["verilog_code", "verilog", "rtl", "code",
                  "rtl_code", "design", "source", "module"]
    for col in candidates:
        if col in df.columns:
            return col
    str_cols = df.select_dtypes(include="object").columns.tolist()
    if str_cols:
        avg_len = {c: df[c].dropna().apply(len).mean() for c in str_cols}
        return max(avg_len, key=avg_len.get)
    return None


def generate_faulty_csv(
    category_key: str,
    input_csv:    str,
    output_csv:   str,
    seed:         int = RANDOM_SEED,
) -> pd.DataFrame:
    """
    Read a clean category CSV, inject all 5 fault types into each row,
    and write the resulting faulty CSV.
    
    The output contains only faulty rows (labels 1-5).  If you also want
    the clean rows (label 0) simply concatenate the original CSV.

    Parameters
    ----------
    category_key : str  e.g. "arithmetic"
    input_csv    : str  path to the clean category CSV
    output_csv   : str  destination path for the faulty CSV
    seed         : int  random seed for reproducibility

    Returns
    -------
    pd.DataFrame  the full faulty dataframe (also written to disk)
    """
    rng = random.Random(seed)

    print(f"\n{'-'*60}")
    print(f"  Category : {category_key}")
    print(f"  Input    : {input_csv}")
    print(f"  Output   : {output_csv}")
    print(f"{'-'*60}")

    if not os.path.exists(input_csv):
        print(f"  [SKIP] Input file not found: {input_csv}")
        return pd.DataFrame()

    df = pd.read_csv(input_csv)
    print(f"  Loaded {len(df)} clean circuits, {len(df.columns)} columns.")

    verilog_col = _find_verilog_column(df)
    if verilog_col is None:
        raise ValueError(f"Could not detect a Verilog code column in {input_csv}.")
    print(f"  Verilog column : '{verilog_col}'")

    faulty_rows: list[dict] = []
    label_counts = {i: 0 for i in range(1, 6)}

    for idx, row in df.iterrows():
        verilog_code = str(row[verilog_col]) if pd.notna(row[verilog_col]) else ""
        if not verilog_code.strip():
            continue

        fault_variants = inject_faults(verilog_code, rng)

        for variant in fault_variants:
            new_row = row.to_dict()                      # copy all original columns
            new_row[verilog_col]   = variant["verilog_code"]
            new_row["fault_type"]  = variant["fault_type"]
            new_row["fault_label"] = variant["fault_label"]
            new_row["fault_site"]  = variant["fault_site"]
            faulty_rows.append(new_row)
            label_counts[variant["fault_label"]] += 1

    faulty_df = pd.DataFrame(faulty_rows)

    os.makedirs(os.path.dirname(output_csv) or ".", exist_ok=True)
    faulty_df.to_csv(output_csv, index=False)

    # ── Summary ───────────────────────────────────────────────────────────────
    label_names = {
        1: "Stuck-at-0",
        2: "Stuck-at-1",
        3: "Delay fault",
        4: "Bridging fault",
        5: "Open fault",
    }
    print(f"\n  Fault-label distribution ({len(faulty_df)} faulty circuits):")
    for lbl, name in label_names.items():
        print(f"    [{lbl}] {name:<18s}: {label_counts[lbl]:5d} samples")
    print(f"\n  Saved → {output_csv}")
    return faulty_df

Main pipeline

In [14]:
def faultInjector(input_dir: str = INPUT_DIR, output_dir: str = OUTPUT_DIR, seed: int = RANDOM_SEED) -> dict[str, pd.DataFrame]:
    """
    Run fault injection for all four circuit categories.

    Parameters
    ----------
    input_dir  : directory containing the clean *_modules.csv files
    output_dir : directory to write the *_faulty_modules.csv files
    seed       : master random seed (each category uses seed + offset)

    Returns
    -------
    dict mapping category key → faulty DataFrame
    """
    os.makedirs(output_dir, exist_ok=True)
    results: dict[str, pd.DataFrame] = {}

    print("-" * 60)
    print("NeuroChip SmartSilicon - Fault Injector")
    print("-" * 60)
    print(f"  Source dir : {input_dir}")
    print(f"  Output dir : {output_dir}")
    print(f"  Random seed: {seed}")

    for offset, (key, fname) in enumerate(CATEGORY_FILES.items()):
        input_csv  = os.path.join(input_dir,  fname)
        output_csv = os.path.join(output_dir, fname.replace("_modules", "_faulty_modules"))
        results[key] = generate_faulty_csv(
            category_key = key,
            input_csv    = input_csv,
            output_csv   = output_csv,
            seed         = seed + offset,
        )

    print("\n" + "-" * 60)
    print("  All categories processed.")
    total = sum(len(df) for df in results.values())
    print(f"  Total faulty circuits generated: {total}")
    return results

---
### 8. Phase 1 Extraction
Circuits: [1,1000]

Clean Circuit Extraction

In [15]:
os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(os.path.join(SAVE_DIR, "arithmetic_PyG"), exist_ok=True)
os.makedirs(os.path.join(SAVE_DIR, "control_logic_PyG"), exist_ok=True)
os.makedirs(os.path.join(SAVE_DIR, "memory_storage_PyG"), exist_ok=True)
os.makedirs(os.path.join(SAVE_DIR, "miscellaneous_PyG"), exist_ok=True)

start_sample = 0

arithmetic_data = build_dataset(
    csv_path = "escad_openrtlset_categorized/arithmetic_modules.csv",
    n_samples = N_SAMPLES,
    sample_start = start_sample,
    save_dir = os.path.join(SAVE_DIR, "arithmetic_PyG")
)

control_logic_data = build_dataset(
    csv_path = "escad_openrtlset_categorized/control_logic_modules.csv",
    n_samples = N_SAMPLES,
    sample_start = start_sample,
    save_dir = os.path.join(SAVE_DIR, "control_logic_PyG")
)

memory_storage_data = build_dataset(
    csv_path = "escad_openrtlset_categorized/memory_storage_modules.csv",
    n_samples = N_SAMPLES,
    sample_start = start_sample,
    save_dir = os.path.join(SAVE_DIR, "memory_storage_PyG")
)

miscellaneous_data = build_dataset(
    csv_path = "escad_openrtlset_categorized/miscellaneous_modules.csv",
    n_samples = N_SAMPLES,
    sample_start = start_sample,
    save_dir = os.path.join(SAVE_DIR, "miscellaneous_PyG")
)

arithmetic_dataset = VLSIFaultDataset(
    root = os.path.join(SAVE_DIR, "arithmetic_PyG"),
    data_list = arithmetic_data
)

control_logic_dataset = VLSIFaultDataset(
    root = os.path.join(SAVE_DIR, "control_logic_PyG"),
    data_list = control_logic_data
)

memory_storage_dataset = VLSIFaultDataset(
    root = os.path.join(SAVE_DIR, "memory_storage_PyG"),
    data_list = memory_storage_data
)

miscellaneous_dataset = VLSIFaultDataset(
    root = os.path.join(SAVE_DIR, "miscellaneous_PyG"),
    data_list = miscellaneous_data
)

print("\n" + "-" * 60)
print("  All categories processed for GNN dataset.")

──────────────────────────────────────────────────
NeuroChip SmartSilicon: GNN Dataset Builder
──────────────────────────────────────────────────
Loading CSV: escad_openrtlset_categorized/arithmetic_modules.csv
      Using rows 0..999
      Loaded 1000 rows, 18 columns
      Columns: ['index', 'ioheader', 'verilog_code', 'full_code', 'llm_response', 'Repo_url', 'lic_name', 'children_index', 'category', 'category_score', 'score_arithmetic', 'score_memory_storage', 'score_control_logic', 'score_miscellaneous', 'is_ambiguous', 'fault_type', 'fault_label', 'fault_site']

Verilog column detected: 'verilog_code'

Parsing Verilog and extracting features (parallel) …
  … processed 50/1000 samples
  … processed 100/1000 samples
  … processed 150/1000 samples
  … processed 200/1000 samples
  … processed 250/1000 samples
  … processed 300/1000 samples
  … processed 350/1000 samples
  … processed 400/1000 samples
  … processed 450/1000 samples
  … processed 500/1000 samples
  … processed 550/1000 

Processing...
Done!
Processing...
Done!
Processing...
Done!
Processing...
Done!


Faulty Circuit Extraction

In [16]:
faultInjector()

FAULTY_CSV_DIR = "escad_openrtlset_faulty"
FAULTY_SAVE_DIR = "VLSI_PYG_Faulty_Set"

os.makedirs(FAULTY_SAVE_DIR, exist_ok=True)
os.makedirs(os.path.join(FAULTY_SAVE_DIR, "arithmetic_faulty_PyG"), exist_ok=True)
os.makedirs(os.path.join(FAULTY_SAVE_DIR, "control_logic_faulty_PyG"), exist_ok=True)
os.makedirs(os.path.join(FAULTY_SAVE_DIR, "memory_storage_faulty_PyG"), exist_ok=True)
os.makedirs(os.path.join(FAULTY_SAVE_DIR, "miscellaneous_faulty_PyG"), exist_ok=True)

arithmetic_faulty_data = build_dataset(
    csv_path = "escad_openrtlset_faulty/arithmetic_faulty_modules.csv",
    n_samples = N_SAMPLES,
    sample_start = start_sample,
    save_dir = os.path.join(FAULTY_SAVE_DIR, "arithmetic_faulty_PyG")
)

control_logic_faulty_data = build_dataset(
    csv_path = "escad_openrtlset_faulty/control_logic_faulty_modules.csv",
    n_samples = N_SAMPLES,
    sample_start = start_sample,
    save_dir = os.path.join(FAULTY_SAVE_DIR, "control_logic_faulty_PyG")
)

memory_storage_faulty_data = build_dataset(
    csv_path = "escad_openrtlset_faulty/memory_storage_faulty_modules.csv",
    n_samples = N_SAMPLES,
    sample_start = start_sample,
    save_dir = os.path.join(FAULTY_SAVE_DIR, "memory_storage_faulty_PyG")
)

miscellaneous_faulty_data = build_dataset(
    csv_path = "escad_openrtlset_faulty/miscellaneous_faulty_modules.csv",
    n_samples = N_SAMPLES,
    sample_start = start_sample,
    save_dir = os.path.join(FAULTY_SAVE_DIR, "miscellaneous_faulty_PyG")
)

arithmetic_faulty_dataset = VLSIFaultDataset(
    root = os.path.join(FAULTY_SAVE_DIR, "arithmetic_faulty_PyG"),
    data_list = arithmetic_faulty_data
)

control_logic_faulty_dataset = VLSIFaultDataset(
    root = os.path.join(FAULTY_SAVE_DIR, "control_logic_faulty_PyG"),
    data_list = control_logic_faulty_data
)

memory_storage_faulty_dataset = VLSIFaultDataset(
    root = os.path.join(FAULTY_SAVE_DIR, "memory_storage_faulty_PyG"),
    data_list = memory_storage_faulty_data
)

miscellaneous_faulty_dataset = VLSIFaultDataset(
    root = os.path.join(FAULTY_SAVE_DIR, "miscellaneous_faulty_PyG"),
    data_list = miscellaneous_faulty_data
)

------------------------------------------------------------
NeuroChip SmartSilicon - Fault Injector
------------------------------------------------------------
  Source dir : escad_openrtlset_categorized
  Output dir : escad_openrtlset_faulty
  Random seed: 42

------------------------------------------------------------
  Category : arithmetic
  Input    : escad_openrtlset_categorized/arithmetic_modules.csv
  Output   : escad_openrtlset_faulty/arithmetic_faulty_modules.csv
------------------------------------------------------------
  Loaded 27521 clean circuits, 18 columns.
  Verilog column : 'verilog_code'

  Fault-label distribution (137605 faulty circuits):
    [1] Stuck-at-0        : 27521 samples
    [2] Stuck-at-1        : 27521 samples
    [3] Delay fault       : 27521 samples
    [4] Bridging fault    : 27521 samples
    [5] Open fault        : 27521 samples

  Saved → escad_openrtlset_faulty/arithmetic_faulty_modules.csv

---------------------------------------------------

Processing...
Done!
Processing...
Done!
Processing...
Done!
Processing...
Done!
